In [1]:
import re
import time
import pandas as pd
import sqlite3
from nltk.corpus import words # English Words
import fastobo # Read .obo Files
from fastobo.doc import OboDoc
import ahocorasick # Searching in String, Fast
from wordfreq import word_frequency # For Excluding Normal Words
from typing import List, Dict, Tuple, Set, Optional, Union, Literal
from IPython.display import clear_output
from english_words import get_english_words_set

In [2]:
en_words = set([*[w.lower() for w in words.words()], "taxa", "pheromone", "pheromones"])
en_words_2 = get_english_words_set(['gcide', 'web2'], lower=True)

In [3]:
conn = sqlite3.connect("COL.db")
cursor = conn.cursor()

In [29]:
# Borrowing Functions
# These functions are from another file, but
# they're going to have a cameo here as some
# names in the .obo files are the names of
# species. I don't want these to be keywords as
# these would incorrectly flag many articles for
# removal for using a scientific name; this is
# bad, naturally.
def found_scientific_name(name):
    return cursor.execute("""
        SELECT NameUsage."col:scientificName"
        FROM NameUsage
        WHERE 
            LOWER(NameUsage."col:scientificName") = LOWER(?)
        LIMIT 1
    """, (name,)).fetchone()

def found_genus(name):
    return cursor.execute("""
        SELECT NameUsage."col:genus"
        FROM NameUsage
        WHERE LOWER("col:genus") = LOWER(?)
        LIMIT 1
    """, (name,)).fetchone()

def re_is_scientific_name(name: str, flags: int = 0) -> bool:
    return re.match(r"^[A-Z][a-z]+\s[a-z]+$", name, flags)

def re_is_scientific_name_abbrev(name: str, flags: int = 0):
    if not re.match(r"^[A-Z]\.\s[a-z]+$", name, flags):
        return False
    return name.split()

def is_scientific_name(name):
    return (re_is_scientific_name(name) and found_scientific_name(name)) or re_is_scientific_name_abbrev(name)

In [30]:
# Is Number
# As far as I know, the regex handles numbers
# formatted with commas, along with negatives
# and decimals.
def is_number(text: str):
    return re.match(r"(^-?((\d{1,3}),)+(\d{3}|\d{3}(.\d+))$|^-?\d{1,3}(.\d+)?$)", text) or text.isnumeric()

In [31]:
# Normal Word
# A word is normal if it can be found in the two
# sets of dictionaries. It is also normal if it is
# considered to be frequent (>= 1e-6). If it's less
# than a character or a number, it is also considered 
# normal.
def normal_word(word: str) -> bool:
    word_clean = word.lower()
    word_clean = word_clean.strip()
    word_clean = re.sub(r'^[^a-z0-9]+|[^a-z0-9]+$', "", word_clean)
    word_clean = word_clean.strip()

    if is_number(word_clean) or len(word_clean) <= 1:
        return True
    
    return bool(
        word_clean in en_words or 
        word_clean in en_words_2 or 
        word_frequency(word_clean, "en") >= 1e-6
    )

In [32]:
# Normal Words
# There are keywords that should not be
# used. For example, "nitrogen" could be found
# in papers we do want. This word is normal;
# compare it to the long word in the cell below,
# for example. Anyway, when given some words, if
# all of the words are normal, it is normal.
def normal_words(words: str) -> bool:
    # Is 1 Character
    if len(words) <= 1:
        return True

    # Is Number
    if is_number(words):
        return True

    # Check Words in Phrase
    is_norm_en_words = [normal_word(word) for word in words.split()]
    return any(is_norm_en_words)

In [33]:
# Extract Words
# In a "word" like "(2R,3S,4S)-1-(4-fluorophenyl)sulfonyl-4-(hydroxymethyl)-3-phenyl-2-azetidinecarbonitrile",
# there are meaningful subwords, like "fluorophenyl" that
# could be (1) used in our search; and (2) match more papers.
# So, we have a function made to extract those subwords.
def extract_words(text: str) -> List[str]:
    return re.findall(r"[a-z]+", text)

In [34]:
# Strip Parenthetical
# This functions removes the parentheticals
# inside of a text. Nested structures are not
# accounted for, unfortunately. However, I do
# not think that this needs to be handled.
def strip_parenthetical(text: str) -> str:
    return re.sub(r"\(.+?\)", "", text)

In [35]:
# Normalize Text
# This function standardizes the text
# so that searching can better take place.
# The text is lowered, whitespaces are
# turned into a single space, and the
# outside whitespace is removed.
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [36]:
# This wi-ll have the keywords that mark
# abstracts for exclusion.
keywords = set()

In [37]:
def names_in_obo(obo_doc: OboDoc) -> List[str]:
    names = []
    for frame in obo_doc:
        if not isinstance(frame, fastobo.term.TermFrame):
            continue
            
        name = None
        for clause in frame:
            if isinstance(clause, fastobo.term.NameClause):
                name = str(clause.name)
                names.append(name)
                break
    return names

In [38]:
do_not_use_as_keywords = []

In [39]:
names = names_in_obo(fastobo.load("./Ontology/chebi_core.obo"))
for name in names:
    if normal_words(name):
        continue

    if is_scientific_name(name):
        do_not_use_as_keywords.append(name)
        do_not_use_as_keywords.extend(name.split())
        continue
    
    keywords.add(name)

    # Try Sub-Words
    words = extract_words(name)
    for word in words:
        if not normal_words(word):
            keywords.add(word)

In [40]:
names = names_in_obo(fastobo.load("./Ontology/pro_reasoned.obo"))
for name in names:  
    if normal_words(name):
        continue

    if is_scientific_name(name):
        do_not_use_as_keywords.append(name)
        do_not_use_as_keywords.extend(name.split())
        continue
    
    # Remove Parenthetical
    # Example: NAC001 (Arabidopsis thaliana) -> NAC001
    if '(' in name and ')' in name:
        name_no_parenthetical = strip_parenthetical(name)
        name_no_parenthetical = name_no_parenthetical.strip()

        if is_scientific_name(name_no_parenthetical):
            do_not_use_as_keywords.append(name_no_parenthetical)
            do_not_use_as_keywords.extend(name_no_parenthetical.split())
            continue
            
        if not normal_words(name_no_parenthetical):
            keywords.add(name_no_parenthetical)
    else:
        keywords.add(name)

In [41]:
names = names_in_obo(fastobo.load("./Ontology/go-basic.obo"))
for name in names:
    if normal_words(name):
        continue

    if is_scientific_name(name):
        do_not_use_as_keywords.append(name)
        do_not_use_as_keywords.extend(name.split())
        continue
    
    keywords.add(name)

In [42]:
do_not_use_as_keywords

['Cricetulus griseus',
 'Cricetulus',
 'griseus',
 'Mesocricetus auratus',
 'Mesocricetus',
 'auratus',
 'Cyberlindnera mrakii',
 'Cyberlindnera',
 'mrakii',
 'Cereibacter sphaeroides',
 'Cereibacter',
 'sphaeroides',
 'Kocuria varians',
 'Kocuria',
 'varians',
 'Enterococcus faecalis',
 'Enterococcus',
 'faecalis',
 'Enterococcus faecium',
 'Enterococcus',
 'faecium',
 'Lactococcus cremoris',
 'Lactococcus',
 'cremoris',
 'Sphingomonas paucimobilis',
 'Sphingomonas',
 'paucimobilis',
 'Aneurinibacillus aneurinilyticus',
 'Aneurinibacillus',
 'aneurinilyticus',
 'Brevibacillus brevis',
 'Brevibacillus',
 'brevis',
 'Heyndrickxia coagulans',
 'Heyndrickxia',
 'coagulans',
 'Priestia megaterium',
 'Priestia',
 'megaterium',
 'Lysinibacillus sphaericus',
 'Lysinibacillus',
 'sphaericus',
 'Geobacillus stearothermophilus',
 'Geobacillus',
 'stearothermophilus',
 'Methanothermobacter wolfeii',
 'Methanothermobacter',
 'wolfeii',
 'Methanothermobacter thermautotrophicus',
 'Methanothermobact

In [43]:
# Add Keywords to Automaton
standardized_keywords = set([normalize_text(keyword) for keyword in keywords])
do_not_use_as_keywords = set([keyword.lower() for keyword in do_not_use_as_keywords])

A = ahocorasick.Automaton()
for keyword in standardized_keywords:
    if keyword in do_not_use_as_keywords:
        continue
    A.add_word(keyword, keyword)

A.make_automaton()

In [44]:
# Show Some Keywords
# This is moreso a smell test. I don't think
# I'll be checking through each and every keyword,
# but perhaps this will provide me some idea.
keys = [*A.keys()]
number_keys = len(keys)
print(f"Number Keys: {number_keys}")

i = 0
step = int(number_keys / 10000)
while i < number_keys:
    print(keys[i])
    i += step

Number Keys: 168735
.alpha.-irone
07h239-a
{(z)-3-(chloromethylene)-2,3-dihydrobenzo[b]oxepin-7-yl}methanol
{2-[1-methyl-3-(trifluoromethyl)-1h-pyrazol-5-yl]phenyl}methanol
jstx-3
jq1
jatrophone
jaceidin 4'-glucuronide
jasmonyl
juglorin
juneol
jbir-76
jbir-92
jbir-81
jbir-102
jbir-05
jmjd6
kxcb
klhl12
kbtbd8
khellactone
ks-619-1
kobiin
kopsan
kcmf1
kdelr2a
kdr i1532t
kdnalpha2-3galbeta1-4glc-ceramide
ku-58050
kytanthine
kynurenine--oxoglutarate transaminase
ki16425
kif11
kijanimicin
kininogen-1
krsb
kappa-carrageenan
kavapyrone
kaempferol-3-rhamnoside-4''-rhamnoside-7-rhamnoside
kanokonol
karetazan-potassium
kaur
kalide
kallikrein-2
kekulene
ketonitrile
ketodihydrosphingosine
keto-3-deoxy-d-manno-octulosonate
ketoprofen
ketotifen
8r-hepe
8alpha-hydroxylabda-13(16),14-dien-19-yl-cis-4-hydroxycinnamate
8(s),15(s)-dihete(1-)
8beta-epoxyangeloyloxy-9alpha-ethoxy-14-oxo-acanthospermolide
8,18-di-hepe
8,11-dihydroxy-delta9-tetrahydrocannabinol
8,9-dihydro-5-hydroxy-8-(1-hydroxy-1-methylethyl

In [46]:
df = pd.read_csv("ScreenedPapers-2.csv")
df.reset_index(inplace=True, drop=True)
print(f"Shape: {df.shape}")

Shape: (32754, 55)


In [47]:
mask = []
errors = []
counts = {None: 0}
keywords_in_paper = {}

t0 = time.time()
texts = df.Abstract.tolist()

for i, text in enumerate(texts):
    try:
        found = False
        for end_index, keyword in A.iter(normalize_text(text)):
            start_index = end_index - len(keyword) + 1

            word_boundary_on_l = start_index == 0 or not text[start_index-1].isalpha()
            word_boundary_on_r = end_index + 1 == len(text) or not text[end_index+1].isalpha()

            if word_boundary_on_l and word_boundary_on_r:
                found = True
                counts[keyword] = counts.get(keyword, 0) + 1
                keywords_in_paper[i] = keywords_in_paper.get(i, [])
                keywords_in_paper[i].append(keyword)
        mask.append(found)
    except Exception as e:
        errors.append((i, e))
        mask.append(None)
        counts[None] += 1

    t1 = time.time()
    elapsed = int(t1 - t0)
    m, s = divmod(elapsed, 60)
    h, m = divmod(m, 60)
    
    clear_output(wait=True)
    print(f"{i+1}/{len(texts)} ({h:d}:{m:02d}:{s:02d})")
    print(f"Number Excluded: {sum([m for m in mask if m != None])}")
    print(f"Number Errors: {counts[None]}")

32754/32754 (0:03:05)
Number Excluded: 4971
Number Errors: 0


In [51]:
counts_items = [*counts.items()]
counts_items.sort(key=lambda count: count[1], reverse=True)
for keyword, count in counts_items:
    print(f"{keyword}: {count}")

zn: 522
glyphosate: 403
igh: 295
tes: 218
eed: 208
ys: 197
atrazine: 153
ene: 143
rop: 142
rac: 142
imidacloprid: 137
nta: 129
olo: 119
reda: 119
carbaryl: 114
chlorpyrifos: 112
sulfide: 110
arg: 100
kairomone: 99
antifungal: 95
ien: 92
malathion: 91
thr: 89
glucocorticoid: 88
yp: 88
flavonoids: 87
xa: 86
este: 83
phe: 79
aci: 79
phospholipid: 79
ert: 76
endosulfan: 75
2,4-d: 73
epn: 72
allelochemical: 66
exa: 64
rst: 62
enta: 58
rimen: 58
dde: 58
fluoxetine: 58
eter: 57
ept: 57
cypermethrin: 57
tebuconazole: 57
rapa: 56
arge: 54
archaea: 54
neonicotinoid: 54
psii: 53
cardenolide: 52
iridoid: 52
yc: 51
aph: 51
photosystem: 51
thiamethoxam: 51
cardenolides: 50
echolocation: 50
cta: 47
biomarker: 47
llo: 45
acto: 44
benomyl: 44
polyunsaturated: 44
carbendazim: 44
monoterpene: 43
allo: 43
ddt: 42
divi: 42
rass: 42
triclosan: 42
flavonoid: 41
fh: 41
ciprofloxacin: 41
tetracycline: 40
fipronil: 40
meja: 40
lw: 39
isc: 39
eno: 38
jasmonate: 38
dica: 37
gly: 36
yl: 36
pn: 36
rans: 35
trien: 3

In [ ]:
# df_excluded = df[mask]
# print(f"Excluded Shape: {df_excluded.shape}")

In [ ]:
# df_included = df[[not _ for _ in mask]]
# print(f"Included Shape: {df_included.shape}")

In [ ]:
# for i, flagged in enumerate(mask):
#     if flagged and df.loc[i, "Score"] == 3:
#         print(df.loc[i, "Abstract"])
#         print("Keywords in Paper:")
#         print(keywords_in_paper[i])
#         print()

In [50]:
# Save Screen Data
df[[not flag for flag in mask]].to_csv("ScreenedPapers-3.csv", index=False, encoding="utf-8")